# ResNet癫痫发作预警

## 实验设计

简单复习下面内容

<div align="center" style="display: flex; justify-content: center; gap: 20px;">

<img src="./images/机器学习类别.png" width="40%" />

<img src="./images/三要素.png" width="40%" />

</div>

ResNet 论文链接：https://arxiv.org/abs/1512.03385


## 数据准备

训练集已经准备好: ./dataset/train
测试集需要小组为单位完成: ./dataset/test

## 代码实现

### 一、导入库和环境配置
运行前先检查库是否都下载过

pip install opencv-python numpy torch torchvision pillow tensorboard 

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import random_split

# 设置随机种子以保证结果可复现 (可选但推荐)
torch.manual_seed(42)
np.random.seed(42)

# 检查是否有可用的 GPU，如果没有则退回 CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)
print(f"当前使用的计算设备: {device}")

### 二、全局超参数与路径配置

In [ ]:
# 指向包含所有类别（未发病 n, 发病 y）的数据文件夹
DATA_DIR = './dataset/train'                  
LOG_DIR = './logs/epilepsy_experiment'  
WEIGHTS_DIR = './weights'               

os.makedirs(WEIGHTS_DIR, exist_ok=True)

# 超参数设置
NUM_FRAMES = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
EPOCHS = 20
NUM_CLASSES = 2

# 验证集划分参数：验证集占数据集的比例
VAL_SPLIT = 0.2

### 三、构建 Dataset 和 DataLoader

In [ ]:
class EpilepsyVideoDataset(Dataset):
    def __init__(self, root_dir, transform=None, num_frames=10):
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames
        # 严格定义类别及其对应的索引: 0 为未发病, 1 为发病
        self.classes = ['n', 'y'] 
        self.data = []

        # 遍历目录，将视频路径和对应的类别标签存入列表
        for cls_idx, cls_name in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls_name)
            if not os.path.exists(cls_path):
                continue # 如果文件夹不存在则跳过
            for vid_name in os.listdir(cls_path):
                if vid_name.endswith(('.mp4', '.avi', '.mov')):
                    self.data.append((os.path.join(cls_path, vid_name), cls_idx))

    def __len__(self):
        # 返回数据集中的视频总数
        return len(self.data)

    def _load_video(self, video_path):
        """核心读取函数：读取视频并均匀采样 num_frames 帧"""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # 使用 linspace 计算均匀采样的帧索引 (例如 100 帧里取 10 帧，大致是 0, 11, 22... 99)
        if total_frames > 0:
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
        else:
            indices = np.zeros(self.num_frames, dtype=int) # 容错处理：空视频
            
        frames = []
        for i in range(total_frames):
            ret, frame = cap.read()
            if not ret: 
                break
            # 如果当前帧的索引在我们计算好的采样点里，则保留
            if i in indices:
                # OpenCV 默认读取为 BGR，转换为 PyTorch 习惯的 RGB
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = Image.fromarray(frame)
                
                # 执行图像预处理 (缩放、归一化等)
                if self.transform:
                    frame = self.transform(frame)
                frames.append(frame)
        cap.release()
        
        # 容错处理：如果视频过短导致采样帧数不足，用最后一张图进行填充 (Padding)
        while len(frames) < self.num_frames:
            frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))
            
        # 将 list 中的帧沿着新维度拼接，形状变为 [Frames, Channels, Height, Width]
        return torch.stack(frames[:self.num_frames]) 

    def __getitem__(self, idx):
        vid_path, label = self.data[idx]
        video_tensor = self._load_video(vid_path)
        return video_tensor, label


In [ ]:

data_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. 实例化总的数据集对象
full_dataset = EpilepsyVideoDataset(DATA_DIR, transform=data_transforms, num_frames=NUM_FRAMES)

# 2. 计算训练集和验证集的具体数量
total_size = len(full_dataset)
val_size = int(total_size * VAL_SPLIT)
train_size = total_size - val_size

# 3. 使用 random_split 划分数据集
# 传入一个固定 seed 的 generator，这样每次运行 Notebook 划分出的集合是固定的，方便排查问题
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

# 4. 将划分好的子集包装成 DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"总视频数: {total_size} | 训练集分配: {len(train_dataset)} | 验证集分配: {len(val_dataset)}")

### 四、手搓 ResNet-18 架构

<div align="center">
<img src=./images/模型结构.png width=40% />
</div>

In [ ]:
class BasicBlock(nn.Module):
    """ResNet-18 的基本构建块"""
    expansion = 1  # ResNet-18 的输出通道数与中间卷积层相同

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        # 第一层 3x3 卷积 (stride=2 时在这里进行下采样)
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        
        # 第二层 3x3 卷积
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        
        # 捷径分支 (Shortcut) 的下采样层
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # 如果输入和输出维度/分辨率不匹配，对 identity 进行下采样适配
        if self.downsample is not None:
            identity = self.downsample(x)

        # 残差连接
        out += identity
        return self.relu(out)


class VideoResNet18(nn.Module):
    """适配视频输入的 2D ResNet-18"""
    def __init__(self, num_classes=2, num_frames=10):
        super().__init__()
        self.inplanes = 64
        self.num_frames = num_frames
        
        # ---------- 基础的 2D CNN 骨干网络 ----------
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # ResNet-18 的四个关键层，堆叠的 block 数量均为 2
        self.layer1 = self._make_layer(BasicBlock, 64, 2)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # ---------- 视频分类特有的部分 ----------
        # ResNet-18 最后一层的特征维度是 512 * 1 = 512
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        # 当步长不为 1 或 输入维度不等于 输出维度时，需要下采样
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )
        
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
            
        return nn.Sequential(*layers)

    def forward(self, x):
        # 1. 输入数据维度: [Batch, Time, Channels, Height, Width]
        batch_size, time_steps, C, H, W = x.size()
        
        # 2. 合并 Batch 和 Time
        x = x.view(batch_size * time_steps, C, H, W)

        # 3. 经过 2D ResNet 提取特征
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # 维度变为: [Batch * Time, 512]
        
        # 4. 时间维度聚合
        x = x.view(batch_size, time_steps, -1)
        x = torch.mean(x, dim=1)  # 对 Time 维度取平均
        
        # 5. 送入分类头
        return self.fc(x)

# 实例化
model = VideoResNet18(num_classes=NUM_CLASSES, num_frames=NUM_FRAMES).to(device)

### 五、训练和验证流程

In [ ]:
# 定义损失函数 (交叉熵) 和优化器 (Adam)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 初始化 TensorBoard Writer
writer = SummaryWriter(LOG_DIR)

print("开始训练...")
for epoch in range(EPOCHS):
    # ---------- 训练阶段 ----------
    model.train()  # 开启训练模式 (启用 BatchNorm 和 Dropout 等)
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. 梯度清零
        optimizer.zero_grad()
        # 2. 前向传播计算预测值
        outputs = model(inputs)
        # 3. 计算损失
        loss = criterion(outputs, labels)
        # 4. 反向传播计算梯度
        loss.backward()
        # 5. 更新权重
        optimizer.step()
        
        # 统计 loss 和 准确率
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # 计算平均 loss 和 acc
    epoch_train_loss = train_loss / len(train_dataset)
    epoch_train_acc = correct_train / total_train
    
    # 将训练指标写入 TensorBoard
    writer.add_scalar('Loss/Train', epoch_train_loss, epoch)
    writer.add_scalar('Accuracy/Train', epoch_train_acc, epoch)

    # ---------- 验证阶段 ----------
    model.eval()  # 开启评估模式 (冻结 BatchNorm 等)
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # 验证阶段不计算梯度，节省显存并加速
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct_val / total_val
    
    # 将验证指标写入 TensorBoard
    writer.add_scalar('Loss/Validation', epoch_val_loss, epoch)
    writer.add_scalar('Accuracy/Validation', epoch_val_acc, epoch)

    print(f'Epoch [{epoch+1}/{EPOCHS}] | '
        f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | '
        f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')
    # 使用 if 语句判断：每 10 轮执行一次，或者是最后第一轮时执行
    if (epoch + 1) % 10 == 0 or (epoch + 1) == EPOCHS:
        # 保存模型权重
        torch.save(model.state_dict(), os.path.join(WEIGHTS_DIR, f'resnet50_epoch_{epoch+1}.pth'))

# 训练结束，关闭 writer
writer.close()
print("训练完成！权重已保存。")

### 六、监控面板

命令行运行下面命令
启动 TensorBoard 并指定日志目录

tensorboard --logdir ./logs/epilepsy_experiment

## 课后作业
1. 小组为单位，剪辑整理出测试集。是否发病比例均衡，各 10 条或以上的 5 秒视频。画面尽量保留单人，稳定出现在画面中央（可以通过裁剪实现）。
2. 完成 ResNet-34 和 ResNet-50 模型。（ResNet-18 和 ResNet-34 可以共用 BasicBlock()，ResNet-50 的输入尺寸不同，请把 BasicBlock() 改为 BottleNeck()，参照代码实现第四部分的表格）
3. 训练20轮，并各自保存模型第 10 轮和 20 轮权重。
4. 截图展示 tensorboard 可视化的训练损失和测试准确率曲线。
5. 仿照训练集代码加载测试集，评估保存的 ResNet-34 和 ResNet-50权重，并print出混淆矩阵。
6. (选做)尝试用预训练权重或者优化模型和训练流程。
<div align="center">
<img src=./images/混淆矩阵.png width=40% />
</div>

每个人提交 notebook，命名为学号+姓名+组号.ipynb。！！只需要组长提交测试集，命名为 组号.zip，包含 n/y 两个文件夹，以及新视频和原视频对应的说明 txt 文件。！！